# AIOps W2-D1: Alert Correlation

Notebook nay load dataset, import cac function trong `correlate.py`, chay pipeline va ghi `results/cluster_summary.json`.

In [1]:
from pathlib import Path
import json

from correlate import build_summary, correlate, load_alerts, load_service_graph, write_summary

BASE_DIR = Path.cwd()
if not (BASE_DIR / "dataset").exists():
    candidate = BASE_DIR / "w2" / "d1"
    if (candidate / "dataset").exists():
        BASE_DIR = candidate
ALERTS_PATH = BASE_DIR / "dataset" / "alerts_sample.jsonl"
SERVICES_PATH = BASE_DIR / "dataset" / "services.json"
OUTPUT_PATH = BASE_DIR / "results" / "cluster_summary.json"

print("Ready:", ALERTS_PATH.relative_to(BASE_DIR), SERVICES_PATH.relative_to(BASE_DIR))

Ready: dataset/alerts_sample.jsonl dataset/services.json


In [2]:
alerts = load_alerts(ALERTS_PATH)
graph = load_service_graph(SERVICES_PATH)

print("Input alerts:", len(alerts))
print("Graph nodes:", len(graph))

Input alerts: 20
Graph nodes: 14


In [3]:
clusters = correlate(alerts, graph, gap_sec=120, max_hop=2)
summary = build_summary(alerts, clusters)
{k: summary[k] for k in ["input_alerts", "output_clusters", "reduction_ratio"]}

{'input_alerts': 20, 'output_clusters': 3, 'reduction_ratio': 0.85}

In [4]:
summary = write_summary(ALERTS_PATH, SERVICES_PATH, OUTPUT_PATH, gap_sec=120, max_hop=2)
print("Wrote", OUTPUT_PATH.relative_to(BASE_DIR))

Wrote results/cluster_summary.json


In [5]:
for cluster in summary["clusters"]:
    services = ", ".join(cluster["services"])
    print(f"{cluster['cluster_id']} count={cluster['alert_count']} services={services} severity={cluster['max_severity']}")

c-000-000 count=18 services=cart-svc, checkout-svc, edge-lb, notification-svc, payment-svc severity=crit
c-000-001 count=1 services=recommender-svc severity=warn
c-000-002 count=1 services=search-svc severity=warn
